## Linear Regression with PySpark 2

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('lin_reg').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 10:17:20 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/26 10:17:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 10:17:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
!curl https://raw.githubusercontent.com/markumreed/data_science_for_everyone/refs/heads/main/pyspark_examples/data/ecommerce.csv >> ecommerce.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 86871  100 86871    0     0   114k      0 --:--:-- --:--:-- --:--:--  114k


In [3]:
df = spark.read.csv('ecommerce.csv', inferSchema=True, header=True)

In [4]:
df.printSchema()

root
 |-- Email: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Avatar: string (nullable = true)
 |-- Avg Session Length: double (nullable = true)
 |-- Time on App: double (nullable = true)
 |-- Time on Website: double (nullable = true)
 |-- Length of Membership: double (nullable = true)
 |-- Yearly Amount Spent: double (nullable = true)



In [5]:
df.head()

Row(Email='mstephenson@fernandez.com', Address='835 Frank TunnelWrightmouth, MI 82180-9605', Avatar='Violet', Avg Session Length=34.49726772511229, Time on App=12.65565114916675, Time on Website=39.57766801952616, Length of Membership=4.0826206329529615, Yearly Amount Spent=587.9510539684005)

#### Setup DataFrame for ML

In [6]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

In [7]:
df.columns

['Email',
 'Address',
 'Avatar',
 'Avg Session Length',
 'Time on App',
 'Time on Website',
 'Length of Membership',
 'Yearly Amount Spent']

In [21]:
coef_var = ['Avg Session Length', 'Time on App', 'Time on Website', 'Length of Membership']

assembler = VectorAssembler(inputCols = coef_var,
                           outputCol = 'features')

In [22]:
output = assembler.transform(df)

In [23]:
final_df = output.select('features', 'Yearly Amount Spent')

In [24]:
train_data, test_data = final_df.randomSplit([0.7, 0.3])

In [25]:
train_data.describe().show()

+-------+-------------------+
|summary|Yearly Amount Spent|
+-------+-------------------+
|  count|                345|
|   mean|  498.0145314957455|
| stddev|  78.56061008841439|
|    min| 256.67058229005585|
|    max|  765.5184619388373|
+-------+-------------------+



In [26]:
test_data.describe().show()

+-------+-------------------+
|summary|Yearly Amount Spent|
+-------+-------------------+
|  count|                155|
|   mean| 502.20648879524555|
| stddev|  81.15074871753771|
|    min| 298.76200786180766|
|    max|  700.9170916173961|
+-------+-------------------+



In [27]:
from pyspark.ml.regression import LinearRegression

In [28]:
lm = LinearRegression(labelCol="Yearly Amount Spent")

In [29]:
model = lm.fit(train_data)

26/06/26 10:31:16 WARN Instrumentation: [2acb1dc1] regParam is zero, which might cause numerical instability and overfitting.


In [30]:
import pandas as pd

In [31]:
pd.DataFrame({'Coefficients': model.coefficients}, index=coef_var)

,Coefficients
Avg Session Length,26.341673
Time on App,39.005623
Time on Website,0.748362
Length of Membership,61.513054


In [32]:
res = model.evaluate(test_data)

In [33]:
res.residuals.show()

+-------------------+
|          residuals|
+-------------------+
| -9.527350431364653|
| 12.439687372396861|
|-2.5049767114723522|
|   8.82072340247555|
|-2.0339913216760124|
| 23.747227582061214|
|  5.258248975834817|
| -4.068045621214992|
| 3.8812920297985443|
|  19.29761021447689|
| -0.594922165189189|
|  8.853182385692833|
| -8.556840556936265|
|-1.4162084477869712|
| -4.777578902682876|
|  9.454480769352472|
|-3.9386663186775195|
| 1.2079772379769906|
|   18.5687751187113|
| 17.832085585491257|
+-------------------+
only showing top 20 rows


In [34]:
unlabeled_data = test_data.select('features')

In [35]:
predictions = model.transform(unlabeled_data)

In [36]:
predictions.show()

+--------------------+------------------+
|            features|        prediction|
+--------------------+------------------+
|[30.3931845423455...|329.45622023455826|
|[30.7377203726281...|  449.341054823833|
|[30.8794843441274...|492.71157669632703|
|[30.9716756438877...|485.81788635441717|
|[31.2681042107507...|425.50452449549994|
|[31.2834474760581...| 568.0338618436062|
|[31.3584771924370...| 489.9177014736406|
|[31.5257524169682...| 448.0336724310969|
|[31.5316044825729...|  432.634313699564|
|[31.6005122003032...|459.87524127662005|
|[31.7216523605090...|348.37184879706183|
|[31.8209982016720...| 415.8220986275205|
|[31.8279790554652...| 448.5595881038778|
|[31.8627411090001...| 557.7143496218337|
|[31.9453957483445...| 661.7975028403348|
|[31.9549038566348...| 430.5433991705745|
|[31.9673209478824...| 449.6885075583298|
|[32.0047530203648...|462.53800388265245|
|[32.0180740106320...|  339.214335626604|
|[32.0478146331398...|479.55747217335215|
+--------------------+------------

In [37]:
print('MAE: ', res.meanAbsoluteError)
print('MSE: ', res.meanSquaredError)
print('RMSE: ', res.rootMeanSquaredError)
print('R2: ', res.r2)
print('Adj R2: ', res.r2adj)

MAE:  7.893814204596516
MSE:  103.41037013928971
RMSE:  10.169088953258779
R2:  0.9841951634887468
Adj R2:  0.9837737011817801
